# 05 · Segment products (clustering, unsupervised)

**Use case:** merchandising has 8,600 books and no taxonomy beyond 'Books' — they want natural segments from titles, descriptions, prices and who reviews them.

**Model / lane:** GraphMAE (unsupervised lane) + k-means on the node embeddings

**Sub-tasks**
1. Define an unsupervised **clustering** task on `product`
2. Train GraphMAE (self-supervised on the relational graph) and cluster the embeddings
3. Read what the run reports (reconstruction `val_loss`, epochs)
4. Assign a sample of products to their clusters and look at the segments

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "product-clusters", kind="data_science")
before = credits_used(ls)

reusing project 44eb67a5-aea9-427e-be04-5efeafe0b708 (amazon-reviews-product-clusters, status=ready)
project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


In [3]:
model = train_or_reuse(p, "Group products into natural segments based on their title, description, price and how they are reviewed",
                       task_type="unsupervised", subtask_type="clustering", enable_text_embedding=True)
metrics = model["metrics"]
show(metrics)
schema = ls.predict.model(model["model_id"])
print("unsupervised type:", schema.get("unsupervised_type"), "· entity:", schema.get("entity_table"))

reusing model fef7e68f-e85f-49ce-ac60-a59a8447d457 (unsupervised, trained 2026-09-15T08:55:34.435207+00:00)
  epoch                  100
  val_loss               0.7757
unsupervised type: clustering · entity: product


GraphMAE is self-supervised: the number the run reports is the masked-reconstruction **validation loss** (lower = the embedding explains the graph better), not a labelled accuracy. The segments themselves are the result — score a sample of products (50 credits each, so 20 here) and read them by their members.

In [4]:
import pandas as pd
product = p.table("product").to_pandas()
sample = product.sample(20, random_state=7)
ids = sample["product_id"].tolist()
batch = ls.predict.predict_batch(model_id=model["model_id"], entity_ids=ids)
sample = sample.assign(cluster=[pr.get("cluster", pr.get("prediction")) for pr in batch["predictions"]])
print("cluster sizes in the sample:", sample["cluster"].value_counts().to_dict())
for cl, grp in sample.groupby("cluster"):
    print(f"\ncluster {cl} · median price {grp['price'].median():.2f}")
    for t in grp["title"].head(4): print("   ", t[:80])

cluster sizes in the sample: {0: 18, 1: 2}

cluster 0 · median price 12.88
    Reaper (Boston Underworld) (Volume 2)
    Rick Steves Snapshot Hill Towns of Central Italy: Including Siena &amp; Assisi
    Halley's Bible Handbook with the New International Version---Deluxe Edition
    Charlotte's Web (50th Anniversary Retrospective Edition)

cluster 1 · median price 11.04
    The Imposter Bride: A Novel
    The Risen: Courage


In [5]:
charged = credits_used(ls) - before
save_metrics(".", {"notebook": "05_product_clustering", "task": "clustering · product (unsupervised)", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"], "training_duration_sec": model.get("training_duration_sec"),
                   "credits_charged_this_run": charged,
                   "headline": {"val_loss": metrics.get("val_loss"), "epochs": metrics.get("epoch"), "clusters_in_sample": sample["cluster"].nunique(), "note": "self-supervised: reconstruction loss, no labels"}})

wrote results/metrics.json


PosixPath('results/metrics.json')